In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!pip geopandas spreg libpysal esda mgwr -q
!pip install contextily -q

In [ ]:
with open('/usr/local/lib/python3.12/dist-packages/mgwr/utils.py', 'r+') as f:
  d = f.readlines()
  f.seek(0)
  for i in d:
    if 'plt.register_cmap(cmap=new_cmap)' not in i:
      f.write(i)
  f.truncate()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.style as mplstyle

from tqdm.auto import tqdm
tqdm.pandas()

from spreg import OLS, diagnostics
from libpysal.weights import Queen
from esda.moran import Moran
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
from mgwr.utils import shift_colormap, truncate_colormap

mpl.rcParams['path.simplify_threshold'] = 1
mplstyle.use('fast')

In [ ]:
census = pd.read_csv('/content/drive/MyDrive/Code/Diploma/Data/98-401-X2016045_English_CSV_data.csv', low_memory=False)
nhs = pd.read_csv('/content/drive/MyDrive/Code/Diploma/Data/99-004-XWE2011001-511.csv', header=1, low_memory=False, encoding='cp863')
fed = gpd.read_file('/content/drive/MyDrive/Code/Diploma/Data/lfed000b21a_e.shp')
opinion = pd.read_excel('/content/drive/MyDrive/Code/Diploma/Data/Mildenberger et al. 2016 _ Distribution of Canadian Climate Opinions.xlsx')
elections = pd.read_csv('/content/drive/MyDrive/Code/Diploma/Data/table_tableau12.csv')

In [ ]:
fed = fed.rename(columns={'FEDUID': 'code'})
fed = fed.astype({'code': 'int64'})
fed = fed.drop(columns=[
    'DGUID',
    'FEDENAME',
    'FEDFNAME',
    'LANDAREA',
    'PRUID'
])

In [ ]:
def get_party(x):
  x = x.split('/')
  first_part = x[0].split(' ')
  last_part = x[1].split(' ')
  party_part = first_part[len(last_part)*-1:]
  if ('Christian' in party_part) or ('CFF' in party_part):
    party_part.remove(party_part[0])
  if ('Climate' in first_part):
    party_part = first_part[-3:]
  return ' '.join(party_part)

elections['party'] = elections['Candidate/Candidat'].apply(lambda x: get_party(x))
elections = elections.rename(columns={
    'Electoral District Number/Numéro de circonscription': 'code',
    'Percentage of Votes Obtained /Pourcentage des votes obtenus': 'votes'
})
elections = elections.drop(columns=[
    'Electoral District Name/Nom de circonscription',
    'Province',
    'Candidate/Candidat',
    'Candidate Residence/Résidence du candidat',
    'Candidate Occupation/Profession du candidat',
    'Votes Obtained/Votes obtenus',
    'Majority/Majorité',
    'Majority Percentage/Pourcentage de majorité'
])
elections = elections[~elections['party'].isin([
    'Animal Protection Party', "CFF - Canada's Fourth Front", 'Christian Heritage Party', 'Communist', 'Independent', 'Libertarian',
    'ML', 'National Citizens Alliance', 'Nationalist', 'No Affiliation', 'PC Party', 'Parti Rhinocéros Party', "People's Party",
    "Pour l'Indépendance du Québec", 'Radical Marijuana', 'Stop Climate Change', 'UPC', 'VCP'
])]
elections = elections.pivot_table(index='code', columns='party', values='votes', aggfunc='sum')
elections = elections.reset_index()
elections = fed.merge(elections, on='code')
elections['EP Manifesto'] = elections.fillna(0).apply(
    lambda x: [x['Bloc Québécois'], x['Conservative'], x['Green Party'], x['Liberal'], x['NDP-New Democratic Party']].index(
        max([x['Bloc Québécois'], x['Conservative'], x['Green Party'], x['Liberal'], x['NDP-New Democratic Party']])
    ),
    axis=1
).map({0: 11.561, 1: 7.381, 2: 10.227, 3: 6.915, 4: 3.571})

In [ ]:
opinion = opinion[(opinion['GeoType']=='Riding')]
opinion = opinion.drop(columns=['GeoType', 'name', 'TotalPop', 'happeningOppose', 'real_parthumanOppose', 'real_humanOppose', 'captrade_binOppose', 'carbon_tax_binOppose'])
opinion = opinion.rename(columns={'GEOID': 'code'})
opinion = fed.merge(opinion, on='code')

In [ ]:
census = census.rename(columns={
    'ALT_GEO_CODE': 'code',
    'GEO_LEVEL': 'level',
    'GEO_NAME': 'name',
    'Member ID: Profile of Federal Electoral Districts (2013 Representation Order) (2247)': 'category_id',
    'DIM: Profile of Federal Electoral Districts (2013 Representation Order) (2247)': 'category',
    'Dim: Sex (3): Member ID: [1]: Total - Sex': 'total',
    'Dim: Sex (3): Member ID: [2]: Male': 'male',
    'Dim: Sex (3): Member ID: [3]: Female': 'female'
})
census['total'] = pd.to_numeric(census['total'], errors='coerce')
census['male'] = pd.to_numeric(census['male'], errors='coerce')
census['female'] = pd.to_numeric(census['female'], errors='coerce')
census = census.drop(columns=[
    'CENSUS_YEAR',
    'GEO_CODE (POR)',
    'GNR',
    'GNR_LF',
    'DATA_QUALITY_FLAG',
    'Notes: Profile of Federal Electoral Districts (2013 Representation Order) (2247)'
])
census = census[census['level']==2]
census = census.set_index('code')

In [ ]:
nhs['Characteristic'] = nhs['Characteristic'].fillna('-')
nhs['Characteristic'] = nhs['Characteristic'].str.strip()
nhs['Total'] = pd.to_numeric(nhs['Total'], errors='coerce')
nhs = nhs.drop(columns=['Prov_Name', 'FED_Name', 'GNR', 'Topic', 'Note', 'Flag_Total', 'Male', 'Flag_Male', 'Female', 'Flag_Female'])
nhs = nhs.rename(columns={'Geo_Code': 'code'})
nhs['code'] = pd.to_numeric(nhs['code'], errors='coerce')
nhs = nhs.dropna(subset=['code'])
nhs['code'] = nhs['code'].astype(int)
nhs = nhs.set_index('code')

In [ ]:
data = elections.merge(opinion.drop(columns=['geometry']), on='code')
data = data.merge(
    (census[census['category_id']==8]['male'] / census[census['category_id']==8]['total'] * 100).reset_index().rename(columns={0: 'gender'}),
    on='code'
)
data = data.merge(
    census[census['category_id']==39]['total'].reset_index().rename(columns={'total': 'age'}),
    on='code'
)
data = data.merge(
    census[census['category_id']==751]['total'].reset_index().rename(columns={'total': 'income'}),
    on='code'
)
data = data.merge(
    (census[census['category_id']==1701]['total'] / census[census['category_id']==1698]['total'] * 100).reset_index().rename(columns={'total': 'education'}),
    on='code'
)
data = data.merge(
    (census[census['category_id']==115]['total'] / census[census['category_id']==113]['total'] * 100).reset_index().rename(columns={'total': 'langauge'}),
    on='code'
)
data = data.merge(
    (census[census['category_id']==1868]['total'] / census[census['category_id']==1865]['total'] * 100).reset_index().rename(columns={'total': 'unemployment'}),
    on='code'
)
data = data.merge(
    (census[census['category_id']==1139]['total'] / census[census['category_id']==1135]['total'] * 100).reset_index().rename(columns={'total': 'immigrants'}),
    on='code'
)
data = data.merge(
    ((1 - (census[census['category_id']==1353]['total'] / census[census['category_id']==1338]['total'])) * 100).reset_index().rename(columns={'total': 'noneuropean'}),
    on='code'
)
data = data.merge(
    (nhs[nhs['Characteristic']=='Catholic']['Total'] / nhs[nhs['Characteristic']=='Total population in private households by religion']['Total'] \
     * 100).reset_index().rename(columns={'Total': 'catholic'}),
    on='code'
)
data = data.merge(
    (census[census['category_id']==8]['total']).reset_index().rename(columns={'total': 'density'}),
    on='code'
)
data['density'] = data['density'] / (data.area / 1000000)

In [ ]:
data.info()

In [ ]:
data['income'] = data['income'] / 1000
data = data.fillna(0)
data = data.rename(columns={
      'age': 'MnAGE',
      'income': 'MnINC',
      'gender': 'PtMAL',
      'education': 'PtEDU',
      'langauge': 'PtENG',
      'unemployment': 'PtUNE',
      'immigrants': 'PtIMM',
      'noneuropean': 'PtNEU',
      'catholic': 'PtCTH',
      'density': 'DnPOP',
      'happening': 'PtHAP',
      'real_parthuman': 'PtRPH',
      'real_human': 'PtRHU',
      'captrade_bin': 'PtCTR',
      'carbon_tax_bin': 'PtCTX'
})

In [ ]:
for col in ['Green Party', 'Liberal', 'Conservative', 'NDP-New Democratic Party']:
  fig = plt.figure(figsize=(21, 9))
  gs = fig.add_gridspec(3, 2, hspace=0.15, width_ratios=[5, 1])

  city_bounds = {
    'Ванкувер': {'xlim': (4.0e6, 4.08e6), 'ylim': (1.96e6, 2.04e6)},
    'Торонто': {'xlim': (7.18e6, 7.28e6), 'ylim': (0.88e6, 0.98e6)},
    'Монтреаль': {'xlim': (7.58e6, 7.68e6), 'ylim': (1.20e6, 1.30e6)}
  }

  ax_main = fig.add_subplot(gs[:, 0])

  cmap = plt.cm.YlGn
  vmin = data[col].min()
  vmax = data[col].max()
  sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))

  data.plot(col, cmap=sm.cmap, ax=ax_main, vmin=vmin, vmax=vmax, **{'edgecolor': 'black', 'linewidth': 0.1})
  ax_main.set_axis_off()
  ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
  ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=data.crs)

  city_axes = {
    'Ванкувер': fig.add_subplot(gs[0, 1]),
    'Торонто': fig.add_subplot(gs[1, 1]),
    'Монтреаль': fig.add_subplot(gs[2, 1])
  }

  for city, ax in city_axes.items():
      ax.set_title(f'{city}', fontsize=14)

      city_data = data.cx[
          city_bounds[city]['xlim'][0]*0.99:city_bounds[city]['xlim'][1]*1.01, city_bounds[city]['ylim'][0]*0.99:city_bounds[city]['ylim'][1]*1.01
      ]

      if not city_data.empty:
          city_data.plot(
              col, cmap=sm.cmap, ax=ax, vmin=vmin, vmax=vmax,
              **{'edgecolor': 'black', 'linewidth': 0.1}
          )
      else:
          ax.text(0.5, 0.5, 'No data in this region', transform=ax.transAxes, ha='center', va='center', color='red')
      ax.set_xlim(city_bounds[city]['xlim'][0], city_bounds[city]['xlim'][1])
      ax.set_ylim(city_bounds[city]['ylim'][0], city_bounds[city]['ylim'][1])
      ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
      ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=10, crs=data.crs)
      ax.set_axis_off()

  cax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
  sm._A = []
  cbar = fig.colorbar(sm, cax=cax)
  cbar.ax.tick_params(labelsize=10)
  cbar.ax.yaxis.set_major_formatter('{x:1.0f}%')
  plt.subplots_adjust(wspace=0, hspace=0.06)
  plt.savefig(f'/content/drive/MyDrive/Code/Diploma/Results/Preview_{col}.png', dpi=300, bbox_inches='tight')
  plt.show()
  plt.close('All')

In [ ]:
for col in ['PtHAP', 'PtRPH', 'PtRHU', 'PtCTR', 'PtCTX']:
  fig = plt.figure(figsize=(21, 9))
  gs = fig.add_gridspec(3, 2, hspace=0.15, width_ratios=[5, 1])

  city_bounds = {
    'Ванкувер': {'xlim': (4.0e6, 4.08e6), 'ylim': (1.96e6, 2.04e6)},
    'Торонто': {'xlim': (7.18e6, 7.28e6), 'ylim': (0.88e6, 0.98e6)},
    'Монтреаль': {'xlim': (7.58e6, 7.68e6), 'ylim': (1.20e6, 1.30e6)}
  }

  ax_main = fig.add_subplot(gs[:, 0])

  cmap = plt.cm.PuBuGn
  vmin = 0
  vmax = 100
  sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))

  data.plot(col, cmap=sm.cmap, ax=ax_main, vmin=vmin, vmax=vmax, **{'edgecolor': 'black', 'linewidth': 0.1})
  ax_main.set_axis_off()
  ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
  ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=data.crs)

  city_axes = {
    'Ванкувер': fig.add_subplot(gs[0, 1]),
    'Торонто': fig.add_subplot(gs[1, 1]),
    'Монтреаль': fig.add_subplot(gs[2, 1])
  }

  for city, ax in city_axes.items():
      ax.set_title(f'{city}', fontsize=14)

      city_data = data.cx[
          city_bounds[city]['xlim'][0]*0.99:city_bounds[city]['xlim'][1]*1.01, city_bounds[city]['ylim'][0]*0.99:city_bounds[city]['ylim'][1]*1.01
      ]

      if not city_data.empty:
          city_data.plot(
              col, cmap=sm.cmap, ax=ax, vmin=vmin, vmax=vmax,
              **{'edgecolor': 'black', 'linewidth': 0.1}
          )
      else:
          ax.text(0.5, 0.5, 'No data in this region', transform=ax.transAxes, ha='center', va='center', color='red')
      ax.set_xlim(city_bounds[city]['xlim'][0], city_bounds[city]['xlim'][1])
      ax.set_ylim(city_bounds[city]['ylim'][0], city_bounds[city]['ylim'][1])
      ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
      ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=10, crs=data.crs)
      ax.set_axis_off()

  cax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
  sm._A = []
  cbar = fig.colorbar(sm, cax=cax)
  cbar.ax.tick_params(labelsize=10)
  cbar.ax.yaxis.set_major_formatter('{x:1.0f}%')
  plt.subplots_adjust(wspace=0, hspace=0.06)
  plt.savefig(f'/content/drive/MyDrive/Code/Diploma/Results/Preview_{col}.png', dpi=300, bbox_inches='tight')
  plt.show()
  plt.close('All')

In [ ]:
corr = data[[
    'PtMAL', 'MnAGE', 'MnINC', 'PtEDU', 'PtENG', 'PtUNE', 'PtIMM', 'PtNEU', 'PtCTH', 'DnPOP', 'PtHAP', 'PtRPH', 'PtRHU', 'PtCTR', 'PtCTX',
]].corr()
corr.round(2).style.background_gradient(cmap='coolwarm').format(precision=2).to_excel('/content/drive/MyDrive/Code/Diploma/Results/Correlations.xlsx')
corr.round(2).style.background_gradient(cmap='coolwarm').format(precision=2)

In [ ]:
dependent_variables = ['Liberal', 'Green Party', 'Conservative', 'NDP-New Democratic Party', 'EP Manifesto']
cilmate_opinion_regressors = ['PtHAP', 'PtRPH', 'PtRHU', 'PtCTR', 'PtCTX']

In [ ]:
for column in corr.columns:
  if ((len(corr[corr[column] > 0.7]) > 1) or (len(corr[corr[column] < -0.7]) > 0)) and (column not in cilmate_opinion_regressors):
    print(column)

In [ ]:
corr = data[['PtMAL', 'MnAGE', 'MnINC', 'PtEDU', 'PtUNE', 'PtIMM', 'PtNEU', 'PtCTH', 'DnPOP', 'PtHAP', 'PtRPH', 'PtRHU', 'PtCTR', 'PtCTX']].corr()
for column in corr.columns:
  if ((len(corr[corr[column] > 0.7]) > 1) or (len(corr[corr[column] < -0.7]) > 0)) and (column not in cilmate_opinion_regressors):
    print(column)

In [ ]:
corr = data[['PtMAL', 'MnAGE', 'MnINC', 'PtEDU', 'PtUNE', 'PtNEU', 'PtCTH', 'PtHAP', 'PtRPH', 'PtRHU', 'PtCTR', 'PtCTX']].corr()
for column in corr.columns:
  if ((len(corr[corr[column] > 0.7]) > 1) or (len(corr[corr[column] < -0.7]) > 0)) and (column not in cilmate_opinion_regressors):
    print(column)

In [ ]:
coords = list(zip(data.centroid.geometry.x, data.centroid.geometry.y))
w = Queen.from_dataframe(data, use_index=False, silence_warnings=True)
w.transform = 'r'

In [ ]:
for dependent in tqdm(dependent_variables, total=len(dependent_variables)):
  ols_final = pd.DataFrame()
  gwr_final = pd.DataFrame()

  for cilmate_opinion_regressor in cilmate_opinion_regressors:
    ols_final.loc[cilmate_opinion_regressor, cilmate_opinion_regressor] = '-'

  for cilmate_opinion_regressor in cilmate_opinion_regressors:
    print(f'{dependent} -> {cilmate_opinion_regressor}')
    main_regressors = ['PtMAL', 'MnAGE', 'MnINC', 'PtEDU', 'PtUNE', 'PtNEU', 'PtCTH']
    for main_regressor in main_regressors:
      ols_final.loc[main_regressor, cilmate_opinion_regressor] = '-'
    main_regressors.insert(0, cilmate_opinion_regressor)

    print(f'Calibration based on AIC')
    AICs = []
    for main_regressor in main_regressors:
      X = data[cilmate_opinion_regressor].values.reshape((-1,1))
      y = data[dependent].values.reshape((-1,1))

      selector = Sel_BW(coords, y, X, fixed=False)
      bw = selector.search()
      GWR_model = GWR(coords, y, X, bw, fixed=False).fit()

      AICs.append(GWR_model.aicc)

    AIC_min = sorted(AICs)[0]
    AICs_final = [AIC_min]
    next_regressor = main_regressors[AICs.index(AIC_min)]
    final_regressors = [next_regressor]
    main_regressors.remove(next_regressor)

    for i in range(len(main_regressor)+1):
      AICs = []
      for main_regressor in main_regressors:
        final_regressors.append(main_regressor)
        X = data[final_regressors].values
        y = data[dependent].values.reshape((-1,1))

        selector = Sel_BW(coords, y, X, fixed=False)
        bw = selector.search()
        GWR_model = GWR(coords, y, X, bw, fixed=False).fit()

        AICs.append(GWR_model.aicc)
        final_regressors.remove(main_regressor)

      if sorted(AICs)[0] > AIC_min:
        break
      AIC_min = sorted(AICs)[0]
      AICs_final.append(AIC_min)
      next_regressor = main_regressors[AICs.index(AIC_min)]
      final_regressors.append(next_regressor)
      main_regressors.remove(next_regressor)

    main_regressors.insert(0, final_regressors[-1])
    AICs.insert(0, AICs_final[-1])

    fig = plt.figure(figsize=(10, 4))
    plt.plot(final_regressors, AICs_final, color='limegreen', label='Выбранные переменные (включительно)')
    plt.plot(main_regressors, AICs, color='red', label='Отброшенные переменные')
    plt.xlabel('Добавленная переменная', fontsize=12)
    plt.ylabel('Текущий минимальный AICc', fontsize=12)
    ax = fig.get_axes()[0]
    ax.spines[['right', 'top']].set_visible(False)
    plt.grid()
    plt.legend(fontsize=8)
    plt.savefig(f'/content/drive/MyDrive/Code/Diploma/Results/AIC_calibration_{dependent}_{cilmate_opinion_regressor}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close('All')

    print(f'Final regressors: {", ".join(final_regressors)}')
    if cilmate_opinion_regressor not in final_regressors:
      continue
    final_regressors.remove(cilmate_opinion_regressor)
    final_regressors.insert(0, cilmate_opinion_regressor)
    X = data[final_regressors].values
    y = data[dependent].values.reshape((-1,1))

    k = len(final_regressors)
    n = len(y)

    print('OLS regression')
    ols_model = OLS(y, X, name_y=dependent, name_x=final_regressors)
    for i, row in ols_model.output.iterrows():
      if row['var_names'] == cilmate_opinion_regressor:
        if row['prob'] < 0.01:
          ols_final.loc[cilmate_opinion_regressor, cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '***'
        elif row['prob'] < 0.05:
          ols_final.loc[cilmate_opinion_regressor, cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '**'
        elif row['prob'] < 0.1:
          ols_final.loc[cilmate_opinion_regressor, cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '*'
        else:
          ols_final.loc[cilmate_opinion_regressor, cilmate_opinion_regressor] = str(round(row['coefficients'], 4))
      elif row['var_names'] == 'CONSTANT':
        if row['prob'] < 0.01:
          ols_final.loc['CONSTANT', cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '***'
        elif row['prob'] < 0.05:
          ols_final.loc['CONSTANT', cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '**'
        elif row['prob'] < 0.1:
          ols_final.loc['CONSTANT', cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '*'
        else:
          ols_final.loc['CONSTANT', cilmate_opinion_regressor] = str(round(row['coefficients'], 4))
      else:
        if row['prob'] < 0.01:
          ols_final.loc[row['var_names'], cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '***'
        elif row['prob'] < 0.05:
          ols_final.loc[row['var_names'], cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '**'
        elif row['prob'] < 0.1:
          ols_final.loc[row['var_names'], cilmate_opinion_regressor] = str(round(row['coefficients'], 4)) + '*'
        else:
          ols_final.loc[row['var_names'], cilmate_opinion_regressor] = str(round(row['coefficients'], 4))

    ols_final.loc['AICc', cilmate_opinion_regressor] = round((ols_model.aic + ((k*k + k)/(n - k - 1))), 0)
    ols_final.loc['R2', cilmate_opinion_regressor] = round(ols_model.r2, 2)
    ols_final.loc['Adj. R2', cilmate_opinion_regressor] = round(ols_model.ar2, 2)

    ols_resids = ols_model.u
    moran = Moran(ols_resids, w)
    ols_final.loc["Moran's I", cilmate_opinion_regressor] = round(moran.I, 3)

    if len(final_regressors) > 1:
      VIF = diagnostics.vif(ols_model)
      max_vif = 0
      for vif in VIF[1:]:
        if vif[0] > max_vif:
          max_vif = vif[0]
      ols_final.loc['Max VIF', cilmate_opinion_regressor] = max_vif


    print('GWR regression')
    selector = Sel_BW(coords, y, X, fixed=False)
    bw = selector.search(criterion='BIC', rss_score=True)
    print(f'Proposed bandwith: {bw}')
    GWR_model = GWR(coords, y, X, bw, fixed=False).fit()

    data[f'GWR_{dependent}_{cilmate_opinion_regressor}'] = GWR_model.params[:, 1]
    data[f'GWR_R2_{dependent}_{cilmate_opinion_regressor}'] = GWR_model.localR2
    gwr_filtered_t = GWR_model.filter_tvals()
    gwr_final = pd.concat([gwr_final, data[f'GWR_{dependent}_{cilmate_opinion_regressor}'].describe().round(4)], axis=1)
    gwr_final = gwr_final.rename(columns={f'GWR_{dependent}_{cilmate_opinion_regressor}': cilmate_opinion_regressor})

    gwr_final.loc['AICc', cilmate_opinion_regressor] = round(GWR_model.aicc, 0)
    gwr_final.loc['R2', cilmate_opinion_regressor] = round(GWR_model.R2, 2)
    gwr_final.loc['Adj. R2', cilmate_opinion_regressor] = round(GWR_model.adj_R2, 2)

    gwr_resids = GWR_model.resid_response
    moran = Moran(gwr_resids, w)
    gwr_final.loc["Moran's I", cilmate_opinion_regressor] = round(moran.I, 3)

    gwr_final.loc['BW', cilmate_opinion_regressor] = bw
    gwr_p_values_stationarity = GWR_model.spatial_variability(selector, 1000)
    gwr_final.loc['Monte Carlo (p-value)', cilmate_opinion_regressor] = gwr_p_values_stationarity[1]

    if len(final_regressors) > 1:
      VCC, VIF, CN, VDP = GWR_model.local_collinearity()
      gwr_final.loc['Max VIF', cilmate_opinion_regressor] = round(max(pd.DataFrame(VIF).max()), 3)
      print(f'Max VIF: {max(pd.DataFrame(VIF).max())}')

    fig = plt.figure(figsize=(21, 9))
    gs = fig.add_gridspec(3, 2, hspace=0.15, width_ratios=[5, 1])

    city_bounds = {
      'Ванкувер': {'xlim': (4.0e6, 4.08e6), 'ylim': (1.96e6, 2.04e6)},
      'Торонто': {'xlim': (7.18e6, 7.28e6), 'ylim': (0.88e6, 0.98e6)},
      'Монтреаль': {'xlim': (7.58e6, 7.68e6), 'ylim': (1.20e6, 1.30e6)}
    }

    ax_main = fig.add_subplot(gs[:, 0])

    col = f'GWR_{dependent}_{cilmate_opinion_regressor}'
    cmap = plt.cm.seismic
    gwr_min = data[col].min()
    gwr_max = data[col].max()
    vmin = np.min([gwr_min])
    vmax = np.max([gwr_max])
    if (vmin < 0) & (vmax < 0):
        cmap = truncate_colormap(cmap, 0.0, 0.5)
    elif (vmin > 0) & (vmax > 0):
        cmap = truncate_colormap(cmap, 0.5, 1.0)
    else:
        cmap = shift_colormap(cmap, start=0.0, midpoint=1 - vmax/(vmax + abs(vmin)), stop=1.)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))

    data.plot(col, cmap=sm.cmap, ax=ax_main, vmin=vmin, vmax=vmax, **{'edgecolor': 'black', 'linewidth': 0.1})
    nonsig_mask = (gwr_filtered_t[:, 1] == 0)
    data_nonsig_full = data[nonsig_mask]
    if not data_nonsig_full.empty:
        data_nonsig_full.plot(color='lightgrey', ax=ax_main, **{'edgecolor':'black', 'linewidth': 0.1})
    ax_main.set_axis_off()
    ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
    ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=data.crs)

    city_axes = {
      'Ванкувер': fig.add_subplot(gs[0, 1]),
      'Торонто': fig.add_subplot(gs[1, 1]),
      'Монтреаль': fig.add_subplot(gs[2, 1])
    }

    for city, ax in city_axes.items():
        ax.set_title(f'{city}', fontsize=14)

        city_data = data.cx[
            city_bounds[city]['xlim'][0]*0.99:city_bounds[city]['xlim'][1]*1.01, city_bounds[city]['ylim'][0]*0.99:city_bounds[city]['ylim'][1]*1.01
        ]
        city_data_nonsig = data_nonsig_full.cx[
            city_bounds[city]['xlim'][0]*0.99:city_bounds[city]['xlim'][1]*1.01, city_bounds[city]['ylim'][0]*0.99:city_bounds[city]['ylim'][1]*1.01
        ]

        if not city_data.empty:
            city_data.plot(
                col, cmap=sm.cmap, ax=ax, vmin=vmin, vmax=vmax,
                **{'edgecolor': 'black', 'linewidth': 0.1}
            )
            if not city_data_nonsig.empty:
                city_data_nonsig.plot(color='lightgrey', ax=ax, **{'edgecolor':'black', 'linewidth': 0.1})
        else:
            ax.text(0.5, 0.5, 'No data in this region', transform=ax.transAxes, ha='center', va='center', color='red')
        ax.set_xlim(city_bounds[city]['xlim'][0], city_bounds[city]['xlim'][1])
        ax.set_ylim(city_bounds[city]['ylim'][0], city_bounds[city]['ylim'][1])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=10, crs=data.crs)
        ax.set_axis_off()

    cax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
    sm._A = []
    cbar = fig.colorbar(sm, cax=cax)
    cbar.ax.tick_params(labelsize=10)
    plt.subplots_adjust(wspace=0, hspace=0.06)
    plt.savefig(f'/content/drive/MyDrive/Code/Diploma/Results/GWR_{dependent}_{cilmate_opinion_regressor}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close('All')

    fig = plt.figure(figsize=(21, 9))
    gs = fig.add_gridspec(3, 2, hspace=0.15, width_ratios=[5, 1])

    city_bounds = {
      'Ванкувер': {'xlim': (4.0e6, 4.08e6), 'ylim': (1.96e6, 2.04e6)},
      'Торонто': {'xlim': (7.18e6, 7.28e6), 'ylim': (0.88e6, 0.98e6)},
      'Монтреаль': {'xlim': (7.58e6, 7.68e6), 'ylim': (1.20e6, 1.30e6)}
    }

    ax_main = fig.add_subplot(gs[:, 0])

    col = f'GWR_R2_{dependent}_{cilmate_opinion_regressor}'
    cmap = plt.cm.PRGn
    vmin = 0
    vmax = 1
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))

    data.plot(col, cmap=sm.cmap, ax=ax_main, vmin=vmin, vmax=vmax, **{'edgecolor': 'black', 'linewidth': 0.1})
    ax_main.set_axis_off()
    ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
    ctx.add_basemap(ax_main, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=data.crs)

    city_axes = {
      'Ванкувер': fig.add_subplot(gs[0, 1]),
      'Торонто': fig.add_subplot(gs[1, 1]),
      'Монтреаль': fig.add_subplot(gs[2, 1])
    }

    for city, ax in city_axes.items():
        ax.set_title(f'{city}', fontsize=14)

        city_data = data.cx[
            city_bounds[city]['xlim'][0]*0.99:city_bounds[city]['xlim'][1]*1.01, city_bounds[city]['ylim'][0]*0.99:city_bounds[city]['ylim'][1]*1.01
        ]

        if not city_data.empty:
            city_data.plot(
                col, cmap=sm.cmap, ax=ax, vmin=vmin, vmax=vmax,
                **{'edgecolor': 'black', 'linewidth': 0.1}
            )
        else:
            ax.text(0.5, 0.5, 'No data in this region', transform=ax.transAxes, ha='center', va='center', color='red')
        ax.set_xlim(city_bounds[city]['xlim'][0], city_bounds[city]['xlim'][1])
        ax.set_ylim(city_bounds[city]['ylim'][0], city_bounds[city]['ylim'][1])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=data.crs)
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=10, crs=data.crs)
        ax.set_axis_off()

    cax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
    sm._A = []
    cbar = fig.colorbar(sm, cax=cax)
    cbar.ax.tick_params(labelsize=10)
    plt.subplots_adjust(wspace=0, hspace=0.06)
    plt.savefig(f'/content/drive/MyDrive/Code/Diploma/Results/GWR_R2_{dependent}_{cilmate_opinion_regressor}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close('All')

  gwr_final = gwr_final.T
  gwr_final = gwr_final.drop(columns=['count', '25%', '75%'])
  gwr_final = gwr_final.rename(columns={'50%': 'median'})

  writer = pd.ExcelWriter(f'/content/drive/MyDrive/Code/Diploma/Results/Regressions_{dependent}.xlsx')
  ols_final.to_excel(writer, sheet_name='OLS')
  gwr_final.to_excel(writer, sheet_name='GWR')
  writer.close()